# PSF-match: Pipeline CLI Demo

This notebook demonstrates every command-line flag of `main.py` through runnable examples. Each code cell executes the pipeline with a specific flag combination - run them selectively as needed.

## Pipeline Overview

PSF-match performs a 9-stage photometric and astrometric calibration of a FITS image against Gaia DR3:

| # | Stage | Description |
|---|-------|-------------|
| 0 | **Loading** | Reads the FITS image, Gaia source catalog, and optional Milliquas quasar catalog |
| 1 | **Source Detection** | Iterative DAOStarFinder with FWHM convergence; estimates background RMS |
| 2 | **PSF Modeling** | Builds parametric (Moffat) and empirical (ePSF) point-spread-function models |
| 3 | **Photometry** | Extracts fluxes using Moffat-fitting, empirical PSF, and aperture methods |
| 4 | **Catalog Matching** | Cross-matches detected sources with proper-motion-corrected Gaia positions |
| 5 | **Systematic Corrections** | Fits SIP polynomial distortion + magnitude-dependent systematics |
| 6 | **Advanced Calibration** | Thin-plate spline (TPS) and Gaussian Process (GP) spatial residual modeling |
| 7 | **Enhancements** | DCR correction, CCD edge diagnostics, holdout cross-validation, magnitude stratification |
| 8 | **Faint-Star Analysis** | Precision analysis stratified by Gaia G magnitude; faint-star local anchoring |

Outputs are written under `output/<instrument>/<variant>/`:
- `catalogs/` --- matched source tables (CSV)
- `models/` --- PSF model files
- `plots/` --- diagnostic figures (PNG)
- `logs/` --- telemetry and precision reports
- `rms/` --- final astrometric RMS (CSV)

## Setup

Choose one method to obtain the project files:

### Option A: Git clone (preferred, once the repo is public)

Run the cell below. It clones the repository from GitHub and enters the project directory.

In [ ]:
# Clone the PSF-match repository (once it is public)
# If already cloned, this cell does nothing.
import os

REPO_URL = "https://github.com/wolf9611/psf-match-astrometry.git"
LOCAL_DIR = "psf-match-astrometry"

if not os.path.exists(LOCAL_DIR):
    !git clone {REPO_URL}
else:
    print(f"'{LOCAL_DIR}' already exists; skipping clone.")

%cd {LOCAL_DIR}


### Option B: Upload ZIP (for private repos or offline use)

If the repository is still private, download it as a ZIP from GitHub (**Code -> Download ZIP**), upload it to Colab with the file browser, then run the extraction cell below. Skip the clone cell above.

### Initialisation

This cell detects your environment (local VS Code, terminal, or Google Colab) and sets the project root. **Important:** If Colab prompts you to restart the session after installing dependencies (below), you must re-run this cell after the restart - otherwise later commands will fail with `FileNotFoundError`.

In [ ]:
# Initialisation: detect environment, set project root, and add to sys.path
# ------------------------------------------------------------------------
# Re-run this cell after every Colab session restart.
# ------------------------------------------------------------------------
import os
import sys
from pathlib import Path

cwd = Path(os.getcwd()).resolve()

# Local Jupyter (VS Code / terminal)
if (cwd / "main.py").exists():
    root = cwd
elif (cwd.parent / "main.py").exists():
    root = cwd.parent
else:
    # Google Colab (remote kernel)
    root = Path("/content/psf-match-astrometry")
    if not (root / "main.py").exists():
        print("=" * 60)
        print("Project not found. If running on Colab:")
        print("1. Run the git clone cell or upload the ZIP file")
        print("2. Run the extraction cell (if using ZIP)")
        print("3. Re-run this cell")
        print("=" * 60)
        raise FileNotFoundError(f"main.py not found in {root}")

os.chdir(root)
sys.path.insert(0, str(root))
print(f"Project root: {root}")
print(f"  main.py exists: {(root / 'main.py').exists()}")


In [ ]:
# Extract the project ZIP (Colab, Option B)
# Change ZIP_NAME if your downloaded file has a different name.
ZIP_NAME = "psf-match-astrometry-main.zip"
!unzip -o {ZIP_NAME} -d /content/psf-match-astrometry


## Installation

Run the install cells **once** before executing any pipeline cells.

In [ ]:
# Install core CPU dependencies
%pip install -r requirements.txt

> **Colab users:** After the `%pip install` completes, Colab will show a **'Restart session'** prompt. Click **Restart** (or go to **Runtime -> Restart session** in the top menu), then **re-run the Initialisation cell** above before continuing. This ensures newly installed packages are loaded correctly.

If you have an NVIDIA GPU and want CUDA acceleration, install the optional GPU dependencies as well. The pipeline runs fine without them.

In [ ]:
# Optional: install GPU dependencies (cupy-cuda12x, numba, torch)
# Only needed if running with --device cuda
%pip install -r requirements-gpu.txt

> **Colab users:** If you installed GPU dependencies for the first time, Colab will ask to restart again. Click **Restart**, then **re-run the Initialisation cell** before running any pipeline commands.

## All Available Flags

The cell below prints the full CLI help for `main.py`, listing every flag and its default value.

In [ ]:
# Show all CLI flags with descriptions and defaults
from main import build_parser

parser = build_parser()
parser.print_help()

### Flag Quick Reference

| Flag | Type | Default | Category | Description |
|------|------|---------|----------|-------------|
| `--preset` / `-p` | `FORS2` | *(none)* | Data | Instrument preset that selects input paths and settings |
| `--image` | path | `input/DECam/image.fits` | Data | Path to input FITS image |
| `--gaia-catalog` | path | auto-detected | Data | Path to Gaia reference catalog CSV |
| `--no-milliquas` | flag | off | Data | Disable Milliquas quasar frame-tie catalog |
| `--instrument` / `-i` | string | auto-detected | Output | Instrument name for output sub-directory |
| `--extension` | int | `0` (auto) | I/O | FITS extension to process (0-indexed) |
| `--no-clean` | flag | off | I/O | Skip cleanup of old output files |
| `--pm-mode` | `damped` / `full` / `none` | `damped` | Astrometry | Proper-motion handling: adaptive damping (G>=17), full amplitude, or epoch-only |
| `--no-faint-stars` | flag | off | Astrometry | Exclude G>=20 sources with strict RUWE<=1.4 |
| `--compare-faint-stars` | flag | off | Astrometry | Run pipeline twice (with/without faint stars) and write comparison report |
| `--device` | `auto` / `cpu` / `cuda` | `auto` | GPU | Execution device selection |
| `--deterministic` | flag | off | GPU | Enable reproducible GPU kernels (seed + cudnn deterministic) |
| `--gpu-batch-size` | int | `4096` | GPU | Batch size for GPU-batched photometry and linear algebra |
| `--gpu-stage` | `STAGE=BOOL` | *(repeatable)* | GPU | Per-stage GPU toggle (detection, psf, photometry, systematic, advanced, enhancements, faint) |
| `--gpu-disable-all` | flag | off | GPU | Force-disable GPU on every stage without changing device |
| `--no-gpu-fallback` | flag | off | GPU | Raise errors instead of silently falling back to CPU |
| `--image-precision` | `float32` / `float64` | `float64` | GPU | Numerical precision for image arrays |
| `--astrometry-precision` | `float32` / `float64` | `float64` | GPU | Numerical precision for astrometric arrays |
| `--gpu-timing` | flag | off | GPU | Log per-stage device selection and timing |
| `--gpu-workers` | int | `1` | GPU | Reserved worker slots for future multi-GPU MEF processing |

---
## Pipeline Runs

Each section below demonstrates a different flag combination. The default instrument is **DECam** (Dark Energy Camera); results go to `output/decam/<variant>/`.

### Run 1: DECam Baseline

Processes the DECam image at `input/DECam/image.fits` with CPU-only execution and adaptive proper-motion damping. That FITS file is not in the repository: download a public DECam frame (see the README) and, if you omit a Gaia CSV, the pipeline downloads Gaia DR3 for the field.

| Flag | Value | What it does |
|------|-------|-------------|
| `--device` | `cpu` | Forces CPU execution on all stages. Remove or set to `cuda` if a GPU is available. |
| `--pm-mode` | `damped` | *(default)* Scales down Gaia proper motions adaptively for stars G >= 17 where PM uncertainties are large. |
| `--extension` | `0` | *(default)* Auto-detects the science extension from the FITS header. |
| `--gpu-timing` | *(flag)* | Logs per-stage device selection. With `--device cpu`, this shows CPU for every stage. |

Output: `output/decam/faint_pm_damped/`


In [ ]:
!python main.py --device cpu --pm-mode damped --extension 0 --gpu-timing

### Run 2: FORS2 Preset

Uses the FORS2 preset, with faint stars excluded. Place your own FORS2 frame at `input/FORS2/eso_fors2_single_01.fits` (ESO Science Archive). The Gaia CSV is not bundled; the pipeline downloads Gaia DR3 when `input/FORS2/gaia_catalog_fors2.csv` is missing.

| Flag | Value | What it does |
|------|-------|-------------|
| `--preset` | `FORS2` | Switches input image, Gaia catalog path, and default settings to the FORS2 instrument. |
| `--device` | `cpu` | Runs on CPU. |
| `--pm-mode` | `damped` | Adaptive proper-motion damping (default). |
| `--no-milliquas` | *(flag)* | Disables the Milliquas quasar frame-tie (the FORS2 preset already sets this internally; shown here for explicitness). |
| `--no-faint-stars` | *(flag)* | Excludes Gaia sources with G >= 20, requires RUWE <= 1.4 for all stars, and drops 2-parameter solutions. Trades precision at the faint end for a cleaner sample. |
| `--no-clean` | *(flag)* | Skips cleanup so old outputs are preserved alongside new ones. |

Output: `output/fors2/nofaint_pm_damped/`


In [ ]:
!python main.py --preset FORS2 --device cpu --pm-mode damped --no-milliquas --no-faint-stars --no-clean

### Run 3: PM Mode = none

Disables proper-motion correction entirely. Only Gaia epoch positions (J2016.0) are used, with no propagation to the observation epoch.

| Flag | Value | What it does |
|------|-------|-------------|
| `--pm-mode` | `none` | Zeroes all proper motions. Useful when Gaia PMs are unreliable (e.g., crowded fields, very faint targets) or when the observation epoch is close to J2016.0. |

**Default behavior:** `--pm-mode damped`

In [ ]:
!python main.py --pm-mode none

### Run 4: PM Mode = full

Applies every star's Gaia proper motion at full amplitude with no damping.

| Flag | Value | What it does |
|------|-------|-------------|
| `--pm-mode` | `full` | Uses full Gaia PMs for all stars regardless of magnitude. Best for bright stars (G < 17) where PM uncertainties are small, but can introduce noise from faint stars with poorly measured PMs. |

**Default behavior:** `--pm-mode damped`

In [ ]:
!python main.py --pm-mode full

### Run 5: Compare Faint Stars

Runs the pipeline twice: once without faint stars (strict RUWE <= 1.4, drop 2-param solutions) and once with them (relaxed RUWE <= 2.5 at G >= 20, retain 2-param solutions).

| Flag | Value | What it does |
|------|-------|-------------|
| `--compare-faint-stars` | *(flag)* | Triggers a paired run. A comparison report is written to `output/decam/comparison/logs/faint_star_comparison_report.md` showing RMS and star counts for both configurations. |

Use this to assess whether including faint stars improves or degrades your astrometric solution.

In [ ]:
!python main.py --compare-faint-stars

### Run 6: Custom Image + Catalog

Overrides the default input paths to process an arbitrary FITS image and its Gaia catalog.

| Flag | Value | What it does |
|------|-------|-------------|
| `--image` | `input/DECam/image.fits` | Overrides the default image path (from config or preset). |
| `--gaia-catalog` | `input/DECam/gaia_catalog.csv` | Overrides the default Gaia catalog path. |
| `--no-milliquas` | *(flag)* | Disables the quasar frame-tie (optional; omit if you have a Milliquas catalog). |

Use this pattern for your own FITS image. The paths above are the expected layout, not files shipped with the repository. Leave the Gaia CSV absent to download Gaia DR3 with astroquery, or pass a catalog you exported from the Gaia archive.


In [ ]:
!python main.py --image input/DECam/image.fits --gaia-catalog input/DECam/gaia_catalog.csv --no-milliquas

### Run 7: GPU: Deterministic Mode

Runs the full pipeline on CUDA with reproducible results.

| Flag | Value | What it does |
|------|-------|-------------|
| `--device` | `cuda` | Forces CUDA execution (error if no GPU is available). Use `auto` to fall back to CPU silently. |
| `--deterministic` | *(flag)* | Enables PyTorch `cudnn.deterministic` and sets a fixed manual seed. Produces bit-for-bit identical results across runs (at a small performance cost). |
| `--gpu-timing` | *(flag)* | Logs per-stage device selection (now showing `cuda` for GPU-capable stages). |
| `--gpu-batch-size` | `8192` | *(default: 4096)* Larger batch for GPU-batched aperture photometry and ridge solves. Reduces kernel-launch overhead on high-memory GPUs. |

**Requires:** NVIDIA GPU + `cupy-cuda12x`, `numba`, `torch` installed (see Installation section).

In [ ]:
!python main.py --device cuda --deterministic --gpu-timing --gpu-batch-size 8192

### Run 8: GPU: Per-Stage Toggle

Enables GPU globally but forces photometry to run on CPU.

| Flag | Value | What it does |
|------|-------|-------------|
| `--device` | `cuda` | All stages default to GPU where supported. |
| `--gpu-stage` | `photometry=false` | Forces the photometry stage to CPU while leaving other stages on GPU. Repeat `--gpu-stage` to toggle multiple stages. Recognised stage names: `detection`, `psf`, `photometry`, `systematic`, `advanced`, `enhancements`, `faint`. |

Use this to isolate a problematic stage or compare GPU vs CPU performance on individual stages.

In [ ]:
!python main.py --device cuda --gpu-stage photometry=false

### Run 9: GPU: Disable All

Forces every stage to CPU without changing the active device.

| Flag | Value | What it does |
|------|-------|-------------|
| `--gpu-disable-all` | *(flag)* | All stages run on CPU, but GPU libraries stay imported (for warmup). Equivalent in effect to `--device cpu`, but allows faster switching back to GPU later in the same session. |

**Compare with:** `--device cpu` (also avoids importing GPU libraries entirely).

In [ ]:
!python main.py --gpu-disable-all

### Run 10: Extension + Skip Cleanup

Processes a different FITS extension and preserves previous outputs.

| Flag | Value | What it does |
|------|-------|-------------|
| `--extension` | `1` | Processes the second FITS extension (0-indexed). Default `0` auto-detects the science extension from the header. Useful for MEF (Multi-Extension FITS) files with multiple image planes. |
| `--no-clean` | *(flag)* | Prevents deletion of previous outputs in the target directory. Useful when re-running with minor config tweaks or comparing multiple runs. |

In [ ]:
!python main.py --extension 1 --no-clean

### Run 11: GPU: Fallback + Precision

CUDA execution with strict error handling and reduced memory precision.

| Flag | Value | What it does |
|------|-------|-------------|
| `--device` | `cuda` | Forces CUDA execution. |
| `--no-gpu-fallback` | *(flag)* | Raises a hard error if any GPU primitive fails, instead of silently falling back to CPU. Use during development or debugging to catch GPU issues early. |
| `--image-precision` | `float32` | *(default: float64)* Uses 32-bit floating point for image arrays. Halves GPU memory usage for image data at the cost of reduced numerical precision. |
| `--astrometry-precision` | `float32` | *(default: float64)* Uses 32-bit floating point for astrometric arrays. Further reduces GPU memory footprint. |
| `--gpu-workers` | `2` | *(default: 1)* Reserves worker slots for future multi-GPU MEF processing. The current implementation remains single-process; values >1 are placeholders. |

Use `float32` precision on GPUs with limited VRAM (<= 4 GB). The astrometric impact is typically < 0.1 mas for well-sampled images.

In [ ]:
!python main.py --device cuda --no-gpu-fallback --image-precision float32 --astrometry-precision float32 --gpu-workers 2

## Results: Compare All Runs

The cell below loads every `final_rms_<variant>.csv` found across all output directories and prints a side-by-side comparison of final astrometric RMS and star count for each configuration.

In [ ]:
import glob

import pandas as pd

rms_files = sorted(glob.glob("output/*/rms/final_rms_*.csv"))
if not rms_files:
    print("No RMS files found. Run some pipeline cells first.")
else:
    print(f"{'Instrument':>10} {'Variant':<45} {'RMS (mas)':>10} {'N Stars':>8}")
    print("-" * 77)
    for f in rms_files:
        df = pd.read_csv(f)
        row = df.iloc[0]
        inst = f.split("/")[1]
        print(
            f"{inst:>10} {row['subdir']:<45} {row['final_rms_mas']:>10.3f} {int(row['n_stars']):>8d}"
        )